# Modelo 2: Predictor de Goles - Biomecánica y Lasso
**Taller 2 - Machine Learning I**

Este notebook desarrolla el modelo de regresión para predecir el volumen total de goles en un partido. El motor del modelo es la **Fatiga Muscular** (días de descanso) y la importancia de las plantillas en el Fantasy League.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from datetime import timedelta
from sklearn.model_selection import KFold, cross_validate
from sklearn.linear_model import LassoCV, RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, make_scorer

sns.set_theme(style="whitegrid")

## 1. Feature Engineering: El Reloj Biológico y la Influencia
Calculamos los días de descanso exactos y las métricas de influencia del Fantasy (FPL) asumiendo la restricción de **Anti-Leakeage**.

In [ ]:
df = pd.read_csv('../../data/matches.csv')
df['date_parsed'] = pd.to_datetime(df['date'], format='%d/%m/%Y')
df = df.sort_values(by=['date_parsed', 'time']).reset_index(drop=True)

match_metrics = ['fthg', 'ast']
team_home_stats = {t: {m: [] for m in match_metrics} for t in df['home_team'].unique()}
team_away_stats = {t: {m: [] for m in match_metrics} for t in df['away_team'].unique()}
team_dates = {t: [] for t in pd.concat([df['home_team'], df['away_team']]).unique()}

historical_feats = []
for idx, row in df.iterrows():
    ht, at = row['home_team'], row['away_team']
    curr_date = row['date_parsed']
    
    # Proporciones históricas
    h_avg_fthg = np.mean(team_home_stats[ht]['fthg']) if team_home_stats[ht]['fthg'] else 1.5
    a_avg_ast = np.mean(team_away_stats[at]['ast']) if team_away_stats[at]['ast'] else 4.0
    
    # Días de descanso (Reloj Biológico)
    h_rest = (curr_date - team_dates[ht][-1]).days if team_dates[ht] else 14
    
    historical_feats.append({
        'Home_Avg_FTHG': h_avg_fthg,
        'Away_Avg_AST': a_avg_ast,
        'Home_Days_Rest': h_rest
    })
    
    # Actualizar estadísticas (Post-partido)
    for m in match_metrics:
        team_home_stats[ht][m].append(row[m])
        team_away_stats[at][m].append(row[m])
    team_dates[ht].append(curr_date)
    team_dates[at].append(curr_date)

df_final = pd.concat([df, pd.DataFrame(historical_feats)], axis=1)

# Inyectar Influencia FPL
players = pd.read_csv('../../data/players.csv')
home_influence = players.groupby('team')['influence'].sum().reset_index()
df_final = pd.merge(df_final, home_influence, left_on='home_team', right_on='team', how='left').rename(columns={'influence': 'Home_Sum_influence'})

features = ['Home_Avg_FTHG', 'Away_Avg_AST', 'Home_Days_Rest', 'Home_Sum_influence']
df_final[['date', 'home_team', 'away_team', 'total_goals'] + features].to_csv('../../data/matches_golden_features.csv', index=False)
print("Dataset Golden V6 exportado.")

## 2. Entrenamiento Penalizado (Lasso CV)
Aplicamos transformación logarítmica al target y buscamos el alpha óptimo.

In [ ]:
X = df_final[features]
y_log = np.log1p(df_final['total_goals'])

kf = KFold(n_splits=5, shuffle=True, random_state=42)
lasso_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LassoCV(alphas=np.logspace(-4, 1, 100), cv=kf, random_state=42))
])

lasso_pipe.fit(X, y_log)
print(f"Mejor Alpha Lasso: {lasso_pipe.named_steps['model'].alpha_:.6f}")

# Pesos del Modelo
weights = pd.DataFrame({'Feature': features, 'Beta': lasso_pipe.named_steps['model'].coef_})
weights.sort_values(by='Beta', ascending=False)

## 3. Validación de Supuestos (Análisis de Residuos)
Comprobamos Normalidad e Homocedasticidad según Gauss-Markov.

In [ ]:
y_pred_log = lasso_pipe.predict(X)
residuals = y_log - y_pred_log

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(residuals, kde=True, ax=ax[0], color='indigo')
ax[0].set_title('Normalidad de Residuos (Distribucion)')
sm.qqplot(residuals, line='45', fit=True, ax=ax[1])
ax[1].set_title('Q-Q Plot')
plt.savefig('../../img/residuos_normalidad.png')
plt.show()

plt.figure(figsize=(8, 5))
sns.scatterplot(x=y_pred_log, y=residuals, alpha=0.5)
plt.axhline(0, color='red', ls='--')
plt.title('Homocedasticidad (Prediccion vs Residual)')
plt.savefig('../../img/residuos_homocedasticidad.png')
plt.show()